In [0]:
%sql
-- ── Step 32: Create catalog and schemas ───────────────────────────
CREATE CATALOG IF NOT EXISTS hr_catalog_divansh_divansh;
USE CATALOG hr_catalog_divansh;
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;



In [0]:
%sql
-- ── Step 34: PII Column Masking ───────────────────────────────────
CREATE OR REPLACE FUNCTION hr_catalog_divansh.silver.mask_pii(value STRING)
RETURNS STRING
RETURN CASE
  WHEN is_account_group_member('pii_hr_access') THEN value
  ELSE NULL
END;

ALTER TABLE hr_catalog_divansh.silver.employees_enriched
  ALTER COLUMN full_name SET MASK hr_catalog_divansh.silver.mask_pii;

ALTER TABLE hr_catalog_divansh.silver.employees_enriched
  ALTER COLUMN email SET MASK hr_catalog_divansh.silver.mask_pii;

-- ── Step 35: Row-level filter (dept managers see only their dept) ──
CREATE OR REPLACE FUNCTION hr_catalog_divansh.silver.dept_row_filter(dept_id STRING)
RETURNS BOOLEAN
RETURN is_account_group_member('hr_admin')
    OR current_user() IN (
        SELECT email FROM hr_catalog_divansh.silver.employees_enriched
        WHERE department_id = dept_id
          AND job_title LIKE '%Manager%'
    );

ALTER TABLE hr_catalog_divansh.silver.employees_enriched
  SET ROW FILTER hr_catalog_divansh.silver.dept_row_filter ON (department_id);

-- ── Step 36: Grant access to analytics group ──────────────────────
GRANT SELECT ON VIEW hr_catalog_divansh.gold.dept_payroll_summary TO `hr_analytics`;